# Modern Sign and Factored Optimisers (C04 Topic 02)

**Citations:** Shazeer & Stern (2018 Adafactor); Anil et al. (2019 SM3); Chen et al. (2023 Lion); Luo et al. (2023 CAME); Pagliardini et al. (2024 AdEMAMix).

### Purpose & Topic Bridges
At frontier scale, optimizer states consume the majority of accelerator memory: standard AdamW allocates $2\times$ parameter state (two FP32 buffers $m_t, v_t$). **Lion** eliminates second moments entirely using a binary sign update $\text{sign}(\beta_1 m_t + (1-\beta_1)g_t)$, cutting state memory by 50%. **Adafactor** factors 2D matrix variance $V \in \mathbb{R}^{R \times C}$ into rank-1 row/column vectors $r \in \mathbb{R}^R, c \in \mathbb{R}^C$, cutting variance memory by $>99\%$. **AdEMAMix** pairs fast ($\beta_1=0.9$) and ultra-slow ($\beta_3=0.9999$) momentum EMAs to maintain multi-thousand-step gradient trajectories.

**Decision Boundary:** Deploy **Lion** when memory-bound on LLMs/ViTs, scaling LR down by $3\text{--}10\times$ and weight decay up by $3\text{--}10\times$. Deploy **Adafactor** for massive 50B+ models where even $1\times$ momentum memory causes OOM. Never reuse AdamW's learning rate directly for Lion without rescaling.

## 2. Mathematical Foundations & Governing Equations

1. **Lion Update:** $u_t = \text{sign}(\beta_1 m_t + (1 - \beta_1) g_t), \quad \theta_{t+1} = \theta_t (1 - \eta \lambda) - \eta u_t, \quad m_{t+1} = \beta_2 m_t + (1 - \beta_2) g_t$.
2. **Adafactor Factored Second Moments:** $r_{t+1} = \hat{\beta}_t r_t + (1 - \hat{\beta}_t) \text{mean}_c(S_t), \quad c_{t+1} = \hat{\beta}_t c_t + (1 - \hat{\beta}_t) \text{mean}_r(S_t)$.
3. **Adafactor Outer Product Reconstruction:** $V_t \approx \frac{r_{t+1} c_{t+1}^T}{\bar{r} + \epsilon_1}, \quad U_t = \frac{G_t}{\sqrt{V_t} + \epsilon_2} / \max(1, \frac{\text{RMS}(U_t)}{d_{\text{clip}}})$.
4. **AdEMAMix Multi-Horizon Mixture:** $m_{1,t+1} = \beta_1 m_1 + (1-\beta_1)g, \quad m_{2,t+1} = \beta_3 m_2 + (1-\beta_3)g$.
5. **AdEMAMix Combined Update:** $u_t = (\hat{m}_{1,t+1} + \alpha \hat{m}_{2,t+1}) / (\sqrt{\hat{v}_{t+1}} + \epsilon)$.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/02_modern_sign_and_factored_optimisers.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class ModernSignOptimisersUnifiedModule(nn.Module):
    def __init__(self, in_dim=8, out_dim=8):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def step_lion(self, p, g, m, lr=0.0001, b1=0.9, b2=0.99, wd=0.1):
        u = torch.sign(b1 * m + (1.0 - b1) * g)
        new_p = p * (1.0 - lr * wd) - lr * u
        new_m = b2 * m + (1.0 - b2) * g
        return new_p, new_m

    def step_adafactor_2d(self, p, g, r, c, step=2, lr=0.001, b2=0.8, eps1=1e-30, eps2=1e-3, clip=1.0, wd=0.01):
        decay = 1.0 - (step ** (-b2))
        s = g * g + eps1
        new_r = decay * r + (1.0 - decay) * torch.mean(s, dim=1)
        new_c = decay * c + (1.0 - decay) * torch.mean(s, dim=0)
        v_approx = (new_r.unsqueeze(1) * new_c.unsqueeze(0)) / (torch.mean(new_r) + eps1)
        u = g / (torch.sqrt(v_approx) + eps2)
        clip_fac = torch.clamp(torch.sqrt(torch.mean(u * u)) / clip, min=1.0)
        new_p = p * (1.0 - lr * wd) - lr * (u / clip_fac)
        return new_p, new_r, new_c

    def step_ademamix(self, p, g, m1, m2, v, step=2, lr=0.001, b1=0.9, b2=0.999, b3=0.9999, alpha=5.0, eps=1e-8, wd=0.01):
        new_m1 = b1 * m1 + (1.0 - b1) * g; new_m2 = b3 * m2 + (1.0 - b3) * g
        new_v = b2 * v + (1.0 - b2) * g * g
        m1_hat = new_m1 / (1.0 - b1**step); m2_hat = new_m2 / (1.0 - b3**step); v_hat = new_v / (1.0 - b2**step)
        u = (m1_hat + alpha * m2_hat) / (torch.sqrt(v_hat) + eps)
        new_p = p * (1.0 - lr * wd) - lr * u
        return new_p, new_m1, new_m2, new_v

model = ModernSignOptimisersUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].to(device).t())
    model.linear.bias.copy_(weights['bias'].to(device))
print('ModernSignOptimisersUnifiedModule initialized successfully.')

In [ ]:
p1 = weights['param_1d'].to(device); g1 = weights['grad_1d'].to(device); m1 = weights['m_1d'].to(device)
p_lion, m_lion = model.step_lion(p1, g1, m1)
err_lion = max((p_lion - weights['p_lion'].to(device)).abs().max().item(), (m_lion - weights['m_lion'].to(device)).abs().max().item())

p2 = weights['param_2d'].to(device); g2 = weights['grad_2d'].to(device)
r_in = weights['r_buf'].to(device); c_in = weights['c_buf'].to(device)
p_ada, r_ada, c_ada = model.step_adafactor_2d(p2, g2, r_in, c_in)
err_ada = max((p_ada - weights['p_adafactor'].to(device)).abs().max().item(), (r_ada - weights['r_adafactor'].to(device)).abs().max().item(), (c_ada - weights['c_adafactor'].to(device)).abs().max().item())

m2 = weights['m2_1d'].to(device); v1 = weights['v_1d'].to(device)
p_adem, m1_adem, m2_adem, v_adem = model.step_ademamix(p1, g1, m1, m2, v1)
err_adem = max((p_adem - weights['p_ademamix'].to(device)).abs().max().item(), (m1_adem - weights['m1_ademamix'].to(device)).abs().max().item(), (m2_adem - weights['m2_ademamix'].to(device)).abs().max().item(), (v_adem - weights['v_ademamix'].to(device)).abs().max().item())

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor error: {err_anchor:.2e}'
print(f'All 4 invariants verified! Lion: {err_lion:.2e}, Adafactor: {err_ada:.2e}, AdEMAMix: {err_adem:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Lion Sign Step': lambda: model.step_lion(p1, g1, m1),
    'Adafactor 2D Factored': lambda: model.step_adafactor_2d(p2, g2, r_in, c_in),
    'AdEMAMix Dual Momentum': lambda: model.step_ademamix(p1, g1, m1, m2, v1),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Divergence from Reusing AdamW Learning Rate in Lion
# Lion updates every parameter coordinate with magnitude exactly +/- 1.0 (signum update).
# Reusing AdamW LR (lr = 1e-3) forces coordinates with tiny noise gradients to jump by +/- 1e-3!
param_init = torch.tensor([1.0], device=device); grad_noisy = torch.tensor([1e-4], device=device)
m_zero = torch.tensor([0.0], device=device)

# Under AdamW LR (1e-3), update is 10x larger than gradient magnitude, exploding step size:
p_bad, _ = model.step_lion(param_init, grad_noisy, m_zero, lr=1e-3, wd=0.0)
delta_bad = abs(param_init.item() - p_bad.item())

# Under calibrated Lion LR (1e-4) and scaled weight decay (0.1), steps remain stable:
p_cal, _ = model.step_lion(param_init, grad_noisy, m_zero, lr=1e-4, wd=0.1)
delta_cal = abs(param_init.item() - p_cal.item())

print(f'[FAILURE DEMONSTRATED] Reused AdamW LR exploded Lion coordinate step: {delta_bad:.5f} (10x larger!)')
print(f'[STABILIZED] Calibrated Lion LR produced controlled update: {delta_cal:.5f}')
assert delta_bad > 5.0 * delta_cal, 'Expected step size explosion under unscaled AdamW LR'
assert abs(delta_cal - 1.1e-4) < 1e-6, 'Calibrated Lion step did not match theoretical expectation'
print('[CORRECT] Modern sign and factored optimisers verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **50% Memory Savings with Signum Updates:** Lion replaces second-moment variance tracking with elementwise signs, halving optimizer state memory while matching or beating AdamW.
2. **Rank-1 Factored Variance Compression:** Adafactor reduces 2D variance matrices from $O(RC)$ to $O(R + C)$, delivering $>99\%$ memory compression on Transformer matrices.
3. **Multi-Horizon Momentum Mixture:** AdEMAMix preserves persistent gradient directions over thousands of steps by combining fast ($\beta=0.9$) and ultra-slow ($\beta=0.9999$) EMAs.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 01_first_order_and_adaptive_optimisers](file:///c:/Users/Liam/Code/AI-Examples/notebooks/04_training_methods_and_optimisers/01_gradient_and_curvature_optimisers/01_first_order_and_adaptive_optimisers.ipynb)